<a href="https://colab.research.google.com/github/LennartRedlich/Capstone-Project-/blob/VanAnh/src/notebooks/Finetune_Distilbert_CV_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers datasets accelerate evaluate scikit-learn

import os, re, json, random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [ ]:
set_seed(42)

ANNOTATED_JSON = "/content/linkedin-cvs-annotated.json"
UNLABELED_JSON  = "/content/linkedin-cvs-not-annotated.json"

MODEL_NAME = "distilbert-base-multilingual-cased"

# ---- Training speed knobs ----
MAX_LEN = 96
TRAIN_EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5
WEIGHT_DECAY = 0.01


In [ ]:
def clean_text(x):
    x = "" if x is None else str(x)
    return re.sub(r"\s+", " ", x).strip()

def flatten_linkedin(link):
    if isinstance(link, dict):
        return " ".join([clean_text(v) for v in link.values() if v is not None])
    return clean_text(link)

def build_text(row):

    pos = clean_text(row.get("position", ""))
    org = clean_text(row.get("organization", ""))
    sd  = clean_text(row.get("startDate", ""))
    ed  = clean_text(row.get("endDate", ""))
    status = clean_text(row.get("status", ""))

    link_txt = flatten_linkedin(row.get("linkedin", ""))

    parts = [pos, org, link_txt, sd, ed, status]
    return " | ".join([p for p in parts if p])


In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

profiles_annot = load_json(ANNOTATED_JSON)
profiles_unlab = load_json(UNLABELED_JSON)

print("Annotated CVs:", len(profiles_annot))
print("Unlabeled CVs:", len(profiles_unlab))

rows = []
for cv in profiles_annot:
    for job in cv:
        rows.append(job)

df = pd.DataFrame(rows)
print("All annotated jobs:", df.shape)

df_train = df[
    (df["status"] == "ACTIVE") &
    df["position"].notna() & (df["position"].astype(str).str.len() > 0) &
    df["department"].notna() &
    df["seniority"].notna()
].copy()

df_train["text"] = df_train.apply(lambda r: build_text(r), axis=1)

print("Trainable ACTIVE jobs:", df_train.shape)
print("Departments:", df_train["department"].nunique(), "| Seniorities:", df_train["seniority"].nunique())
df_train[["text","department","seniority"]].head()


Annotated CVs: 609
Unlabeled CVs: 390
All annotated jobs: (2638, 8)
Trainable ACTIVE jobs: (623, 9)
Departments: 11 | Seniorities: 6


,text,department,seniority
0,Prokurist | Depot4Design GmbH | https://www.li...,Other,Management
1,CFO | Depot4Design GmbH | https://www.linkedin...,Other,Management
2,Betriebswirtin | Depot4Design GmbH | https://w...,Other,Professional
3,Prokuristin | Depot4Design GmbH | https://www....,Other,Management
4,CFO | Depot4Design GmbH | https://www.linkedin...,Other,Management


In [ ]:
# Department label mapping
domain_labels = sorted(df_train["department"].unique().tolist())
domain2id = {lab:i for i, lab in enumerate(domain_labels)}
id2domain = {i:lab for lab, i in domain2id.items()}
df_train["domain_id"] = df_train["department"].map(domain2id)

# Seniority label mapping
sen_labels = sorted(df_train["seniority"].unique().tolist())
sen2id = {lab:i for i, lab in enumerate(sen_labels)}
id2sen = {i:lab for lab, i in sen2id.items()}
df_train["seniority_id"] = df_train["seniority"].map(sen2id)

print("Domain labels:", domain2id)
print("Seniority labels:", sen2id)


Domain labels: {'Administrative': 0, 'Business Development': 1, 'Consulting': 2, 'Customer Support': 3, 'Human Resources': 4, 'Information Technology': 5, 'Marketing': 6, 'Other': 7, 'Project Management': 8, 'Purchasing': 9, 'Sales': 10}
Seniority labels: {'Director': 0, 'Junior': 1, 'Lead': 2, 'Management': 3, 'Professional': 4, 'Senior': 5}


In [ ]:
train_df, val_df = train_test_split(
    df_train,
    test_size=0.2,
    random_state=42,
    stratify=df_train["domain_id"]
)

ds_dom_train = Dataset.from_pandas(train_df[["text","domain_id"]].rename(columns={"domain_id":"label"}))
ds_dom_val   = Dataset.from_pandas(val_df[["text","domain_id"]].rename(columns={"domain_id":"label"}))

ds_sen_train = Dataset.from_pandas(train_df[["text","seniority_id"]].rename(columns={"seniority_id":"label"}))
ds_sen_val   = Dataset.from_pandas(val_df[["text","seniority_id"]].rename(columns={"seniority_id":"label"}))

print(ds_dom_train, ds_sen_train)


Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 498
}) Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 498
})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

ds_dom_train = ds_dom_train.map(tokenize_fn, batched=True)
ds_dom_val   = ds_dom_val.map(tokenize_fn, batched=True)

ds_sen_train = ds_sen_train.map(tokenize_fn, batched=True)
ds_sen_val   = ds_sen_val.map(tokenize_fn, batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }


In [ ]:
model_dom = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(domain2id)
)

args_dom = TrainingArguments(
    output_dir="model_domain",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=TRAIN_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer_dom = Trainer(
    model=model_dom,
    args=args_dom,
    train_dataset=ds_dom_train,
    eval_dataset=ds_dom_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_dom.train()
trainer_dom.evaluate()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2554787468.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_dom = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.633456,0.552000,0.064667
2,1.786300,1.612308,0.552000,0.064667
3,1.786300,1.596874,0.552000,0.064667


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.6334564685821533,
 'eval_accuracy': 0.552,
 'eval_macro_f1': 0.06466729147141519,
 'eval_runtime': 11.299,
 'eval_samples_per_second': 11.063,
 'eval_steps_per_second': 0.708,
 'epoch': 3.0}

In [ ]:
model_sen = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(sen2id)
)

args_sen = TrainingArguments(
    output_dir="model_seniority",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=TRAIN_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer_sen = Trainer(
    model=model_sen,
    args=args_sen,
    train_dataset=ds_sen_train,
    eval_dataset=ds_sen_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_sen.train()
trainer_sen.evaluate()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3353792280.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_sen = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.480392,0.408000,0.158980
2,1.512100,1.225976,0.544000,0.220506
3,1.512100,1.119938,0.600000,0.343397


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.1199378967285156,
 'eval_accuracy': 0.6,
 'eval_macro_f1': 0.34339718521464396,
 'eval_runtime': 11.7676,
 'eval_samples_per_second': 10.622,
 'eval_steps_per_second': 0.68,
 'epoch': 3.0}

In [ ]:
rows_un = []
for i, cv in enumerate(profiles_unlab):
    for job in cv:
        rows_un.append({
            "cv_idx": i,
            "organization": job.get("organization", ""),
            "linkedin": job.get("linkedin", ""),
            "position": job.get("position", ""),
            "startDate": job.get("startDate", ""),
            "endDate": job.get("endDate", ""),
            "status": job.get("status", ""),
        })

df_unlab = pd.DataFrame(rows_un)
df_unlab["text"] = df_unlab.apply(lambda r: build_text(r), axis=1)

print(df_unlab.shape)
df_unlab.head()

(1886, 8)


,cv_idx,organization,linkedin,position,startDate,endDate,status,text
0,0,"Keeping The Books, Bookkeeping",,Bookkeeper,2023-03,None,ACTIVE,"Bookkeeper | Keeping The Books, Bookkeeping | ..."
1,0,Playful Paws,,Co-Owner,2018-11,None,ACTIVE,Co-Owner | Playful Paws | 2018-11 | ACTIVE
2,0,S&R services,,Logistics Officer,2019-09,2024-04,INACTIVE,Logistics Officer | S&R services | 2019-09 | 2...
3,0,ABC Supply Co. Inc.,https://www.linkedin.com/company/abc-supply,Truck driver/ laborer,2019-03,2019-09,INACTIVE,Truck driver/ laborer | ABC Supply Co. Inc. | ...
4,0,MB Railways,,Fuel Driver,2018-03,2019-03,INACTIVE,Fuel Driver | MB Railways | 2018-03 | 2019-03 ...


In [ ]:
from torch.utils.data import DataLoader

def tokenize_only(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

def predict_ids(trainer, texts, batch_size=64):
    ds = Dataset.from_dict({"text": texts})
    ds = ds.map(tokenize_only, batched=True)
    ds.set_format(type="torch", columns=["input_ids","attention_mask"])

    loader = DataLoader(ds, batch_size=batch_size)
    trainer.model.eval()
    device = trainer.model.device

    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = {k:v.to(device) for k,v in batch.items()}
            out = trainer.model(**batch)
            p = out.logits.argmax(dim=-1).cpu().numpy().tolist()
            preds.extend(p)
    return preds

In [ ]:
dom_pred_ids = predict_ids(trainer_dom, df_unlab["text"].tolist(), batch_size=64)
sen_pred_ids = predict_ids(trainer_sen, df_unlab["text"].tolist(), batch_size=64)

df_unlab["pred_domain"] = [id2domain[i] for i in dom_pred_ids]
df_unlab["pred_seniority"] = [id2sen[i] for i in sen_pred_ids]

# Save
OUTFILE = "predictions_finetuned_approach3.csv"
df_unlab.to_csv(OUTFILE, index=False)

print("Saved:", OUTFILE)
df_unlab[["position","organization","status","pred_domain","pred_seniority"]].head(10)

Map:   0%|          | 0/1886 [00:00<?, ? examples/s]

Map:   0%|          | 0/1886 [00:00<?, ? examples/s]

Saved: predictions_finetuned_approach3.csv


,position,organization,status,pred_domain,pred_seniority
0,Bookkeeper,"Keeping The Books, Bookkeeping",ACTIVE,Other,Professional
1,Co-Owner,Playful Paws,ACTIVE,Other,Management
2,Logistics Officer,S&R services,INACTIVE,Other,Professional
3,Truck driver/ laborer,ABC Supply Co. Inc.,INACTIVE,Other,Professional
4,Fuel Driver,MB Railways,INACTIVE,Other,Professional
5,Food Delivery Driver,Sysco,INACTIVE,Other,Professional
6,Regional driver,UPS Freight,INACTIVE,Other,Professional
7,delivery/sales/MIT,Domino's Pizza,INACTIVE,Other,Professional
8,Strategy & Investments,Erste Bank und Sparkasse,ACTIVE,Other,Professional
9,International Event Operations Manager,DO & CO AG,INACTIVE,Other,Professional
